# 12 — Response surface & contours

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/12_rsm.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/12_rsm.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/12_rsm.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Fit a quadratic response surface and reproduce the dashboard's RSM contour at any factor pair, k·σ span, and held-others slice.

**Mirrors.** **RSM** tab → *Output*, *Factor A*, *Factor B*, *k·σ* and *held* sliders.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Quadratic response surface

Fit a 2nd-order polynomial in two factors and visualise the resulting contour. The dashboard's RSM tab does this on a 60×60 grid spanning ±k·σ around the means.


In [ ]:
from cubespec import DEFAULT_CSP, sample_independent, compute_outputs_batch
from cubespec.rsm import fit_quadratic, predict_grid
import numpy as np, matplotlib.pyplot as plt

X = sample_independent(DEFAULT_CSP, n=4_000, seed=1337)
P9 = compute_outputs_batch(X)[:, 2]
model = fit_quadratic(X, P9)
print(f'R² = {model.r2:.4f}')


## 2. Contour for (P4_Fx, P1_dx) at ±3σ


In [ ]:
k = 3
base = np.array([p.mean for p in DEFAULT_CSP.params.values()])
p4 = DEFAULT_CSP.params['P4_Fx']
p1 = DEFAULT_CSP.params['P1_dx']
A, B, Z = predict_grid(
    model, base,
    factor_a=4, factor_b=1,
    span_a=(p4.mean - k*p4.sd, p4.mean + k*p4.sd),
    span_b=(p1.mean - k*p1.sd, p1.mean + k*p1.sd),
    grid=60,
)
fig, ax = plt.subplots(figsize=(6.5, 5))
cf = ax.contourf(A, B, Z, levels=20, cmap='RdYlBu_r')
fig.colorbar(cf, ax=ax, label='P9 (MPa)')
ax.set_xlabel('P4_Fx (N)'); ax.set_ylabel('P1_dx (mm)')
ax.set_title('RSM contour — held-others at μ')
plt.tight_layout(); plt.show()


## 3. Effect of the held-others slider

Re-render the same contour with `P0_rho` held at μ + σ instead of μ. The contour shifts upward, which is exactly what the *held* slider in the dashboard does.


In [ ]:
shifted = base.copy()
shifted[0] = DEFAULT_CSP.params['P0_rho'].mean + DEFAULT_CSP.params['P0_rho'].sd
_, _, Z2 = predict_grid(
    model, shifted,
    factor_a=4, factor_b=1,
    span_a=(p4.mean - k*p4.sd, p4.mean + k*p4.sd),
    span_b=(p1.mean - k*p1.sd, p1.mean + k*p1.sd),
    grid=60,
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, Zi, ttl in zip(axes, (Z, Z2), ('held P0_rho = mu', 'held P0_rho = mu+sigma')):
    cf = ax.contourf(A, B, Zi, levels=20, cmap='RdYlBu_r',
                     vmin=min(Z.min(), Z2.min()), vmax=max(Z.max(), Z2.max()))
    ax.set_xlabel('P4_Fx (N)'); ax.set_title(ttl)
axes[0].set_ylabel('P1_dx (mm)')
fig.colorbar(cf, ax=axes, fraction=0.025); plt.show()


## 4. Try this

Change `factor_a` or `factor_b` to (0, 4) for (P0_rho, P4_Fx) — the mass-density × axial-force surface is the most non-linear pair in the calibrated surrogate.


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
